In [5]:
import os
import cv2
import numpy as np

from utils import get_face_landmarks

data_dir = './data/images'

EMOTIONS = ['Happy', 'Sad', 'Surprised']
emotion_to_label = {emotion.lower(): i for i, emotion in enumerate(EMOTIONS)}

output = []


def augment(image):
    augmented = []

    augmented.append(image)

    # flip
    augmented.append(cv2.flip(image, 1))

    # brightness
    brighter = cv2.convertScaleAbs(image, alpha=1.2, beta=30)
    darker = cv2.convertScaleAbs(image, alpha=0.8, beta=-30)

    augmented.append(brighter)
    augmented.append(darker)

    # zoom
    h, w = image.shape[:2]
    cropped = image[int(0.1*h):int(0.9*h), int(0.1*w):int(0.9*w)]
    cropped = cv2.resize(cropped, (w, h))
    augmented.append(cropped)

    # blur
    blur = cv2.GaussianBlur(image, (5, 5), 0)
    augmented.append(blur)

    return augmented


for subject in sorted(os.listdir(data_dir)):
    subject_path = os.path.join(data_dir, subject)

    if not os.path.isdir(subject_path):
        continue

    for image_name in os.listdir(subject_path):
        image_path = os.path.join(subject_path, image_name)

        emotion_name = image_name.split('.')[0].lower()
        label = emotion_to_label.get(emotion_name)

        if label is None:
            continue

        image = cv2.imread(image_path)
        if image is None:
            continue

        images = augment(image)

        for img in images:
            face_landmarks = get_face_landmarks(img)

            # 👇 العدد الجديد = 936 (بدل 1404)
            if len(face_landmarks) > 0:
                face_landmarks.append(label)
                output.append(face_landmarks)

output = np.asarray(output)
np.savetxt('./data/data.txt', output)

print("Total samples:", len(output))

: 